## Problem set 3, Parts 2 and 3: perpetual youth, from five equations to Reiter's method (template)

Computational Macro (HWS 2026), University of Mannheim

*Team: name both team members here.*

This notebook contains the code for Parts 2 and 3 of the problem set. The economy of session 10 has households
with a constant probability of death, income risk, and a firm that rents their savings as capital. With **CARA**
utility it aggregates exactly: five equations, the PRANC, give the same dynamics as Reiter's method with 801
variables. Part 2 (section 6) solves both and lets you change the grid and the economics. With **CRRA** utility
and a borrowing limit (Part 3, section 7) the aggregation fails, only Reiter's method is left, and the benchmark
is the representative agent of session 9.

The household enters every solver as an **argument** `hh`: its marginal utility, the inverse of it, how its
consumption reacts to its marginal value of assets, its labor endowment in every income state, and whether the
lower end of the asset grid is a borrowing limit. Everything else is written once.

**How to work with this notebook**

- Run the cells from top to bottom. After a restart of the kernel, start again at the top.
- Sections 1 to 5 run as they are. Read them anyway: they are the functions of sessions 9 and 10, and the
  comments say where they changed.
- **Part 2 has no gaps.** You call `solve_pranc` and `solve_reiter` with the parameters the questions ask for.
  Cells that say `# Your code for Question ...` are yours.
- **Part 3 has 7 gaps**, marked `...  # GAP k: ...`. Each is one place where the economics of the CRRA economy
  differs from the CARA economy.
- Each part has **Checks**. Do not move on before they fit.
- Every reported result starts with the header line that the solvers print from `par` and `mpar`.

| Section | Content | Gaps | Question |
| --- | --- | --- | --- |
| 1 | Structures: parameters, the household `hh`, grids | | |
| 2 | Tools of sessions 5 to 10: prices, Jacobians, Klein, impulse responses, grids, Young's method | | |
| 3 | The aggregate economy, a PRANC: `solve_pranc` | | Part 1 |
| 4 | The heterogeneous agent economy, Reiter's method: `solve_reiter` | | |
| 5 | Reporting: tables, figures, the analytical distribution | | |
| 6 | Part 2: the CARA economy | | 2.1 to 2.3 |
| 7 | Part 3: the CRRA economy with a borrowing limit | 7 | 3.1 to 3.4 |

**Notation, problem set to code**

| Problem set | Meaning | Code |
| --- | --- | --- |
| $\varphi$, $\gamma$, $\rho_e$, $\sigma_e$, $\rho_Z$ | survival probability, risk aversion, income process, persistence of TFP | `par.phi`, `par.gamma`, `par.rho_e`, `par.sig_e`, `par.rho_Z` |
| $x = \frac{R}{\varphi} a + w \, y(e)$ | cash on hand, with the labor endowment $y(e)$ | `x`, `gri.y` |
| $\mu_t$, $\nu_t$ | the MPC out of cash on hand and the loading on $e$ in the CARA policy | `mu`, `nu` |
| $v = \frac{R}{\varphi} u'(c)$ | the marginal value of assets on the grid, in logs | `lnv`, an $N_a \times N_e$ table |
| $-u'(c) / u''(c)$ | how consumption moves with the log marginal value | `hh.tolerance(c, par)` |
| $a'(a_i, e_m)$, $c(a_i, e_m)$ | savings and consumption policies on the grid | `a_prime`, `c` |
| $\boldsymbol{\lambda}$, $\boldsymbol{\lambda}^0$ | the histogram and the histogram of the newborns | `lam`, `lam_born` |
| $\mathbf{x}_t = (\boldsymbol{\lambda}_t, K_t, Z_t, \mathbf{v}_t)$ | the variables of Reiter's system, states first | `x`, a vector of length $2 N_a N_e + 1$ |
| $\hat K_t$, $\hat C_t$, $\hat Y_t$ | log deviations from the steady state after the shock | `res["K"]`, `res["C"]`, `res["Y"]` |

### 0. Packages

Nothing new to install: everything is in the course environment since week 5. New from the standard library:
`replace(par, sig_e=0.3)` returns a copy of the frozen dataclass `par` with one field changed.

In [ ]:
# Installs the packages of this notebook into the environment of the running kernel.
%pip install "numpy==2.2.*" "scipy==1.15.*" "matplotlib==3.10.*" "quantecon>=0.7"

In [ ]:
import time
import warnings
from dataclasses import dataclass, replace

import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import quantecon as qe
from scipy.linalg import ordqz, inv, solve
from scipy.stats import norm

np.set_printoptions(precision=4, suppress=True)           # four decimals, no scientific notation, in every print
warnings.filterwarnings("ignore", message="The API of rouwenhorst")   # the notice on every rouwenhorst call, as in week 5

### 1. Structures

`par` holds the economics, `mpar` the numerics, `gri` the grids, as in every notebook of the course. New is
`hh`, the **household**: the five things the solvers have to know about its preferences, its income, and its
constraint. The solvers never write down a utility function themselves. Part 2 passes them `CARA`, defined here.
Part 3 passes them the `CRRA` household that you define there.

In [ ]:
@dataclass(frozen=True)
class EconomicParameters:
    alpha: float = 0.36       # capital share
    delta: float = 0.1        # depreciation rate
    gamma: float = 4.0        # risk aversion: absolute with CARA utility, relative with CRRA utility
    phi: float = 44 / 45      # survival probability: 45 periods of life on average
    rho_e: float = 0.9        # persistence of the income shock e
    sig_e: float = 0.1        # unconditional standard deviation of e
    rho_Z: float = 0.9        # persistence of log TFP
    beta: float = 0.97        # discount factor, calibrated: solve_pranc (CARA) or calibrate_beta (CRRA)
    Zbar: float = 1.0         # steady state TFP, calibrated by solve_pranc


@dataclass(frozen=True)
class NumericalParameters:
    na: int = 80              # asset grid points N_a
    ne: int = 5               # income grid points N_e
    amin: float = -30.0       # lowest asset grid point. CARA: no borrowing limit, the grid only has to reach where households go
    amax: float = 100.0       # highest asset grid point
    crit_pol: float = 1e-11   # the EGM stops when ln v moves by less: Reiter needs a tight steady state
    crit_dist: float = 1e-13  # the distribution iteration stops when lam moves by less
    maxiter: int = 100_000    # iteration cap, the safety net


@dataclass(frozen=True)
class Household:
    name: str                 # printed in the header line
    uprime: object            # marginal utility u'(c), called as uprime(c, par)
    uprime_inv: object        # its inverse: the c with u'(c) = m, called as uprime_inv(m, par)
    tolerance: object         # -u'(c) / u''(c): dc = -tolerance * d ln u'(c), called as tolerance(c, par)
    endowment: object         # labor endowment y(e) in income state e, with mean one: income is w y(e). Called as endowment(e, lam_e)
    borrowing_limit: bool     # True: a' >= a_1 is a constraint of the household. False: a_1 is only the end of the grid


@dataclass(frozen=True)
class Grids:
    a: np.ndarray             # asset grid, N_a
    e: np.ndarray             # grid of the income state e, N_e
    y: np.ndarray             # labor endowment y(e) on that grid, N_e: income is w y
    Pi: np.ndarray            # transition matrix of e, N_e x N_e
    lam_e: np.ndarray         # stationary distribution of e, N_e

In [ ]:
def uprime_cara(c, par):
    # CARA utility u(c) = -exp(-gamma c) / gamma: marginal utility, defined for every real c
    return np.exp(-par.gamma * c)


def uprime_inv_cara(m, par):
    # The c with u'(c) = m
    return -np.log(m) / par.gamma


def tolerance_cara(c, par):
    # -u'(c) / u''(c) = 1 / gamma: the same at every level of consumption
    return np.full_like(c, 1 / par.gamma)


def endowment_cara(e, lam_e):
    # Income is w (1 + e), with e an AR(1) in levels. lam_e, the stationary distribution of e, is not needed here
    return 1 + e


CARA = Household("CARA", uprime_cara, uprime_inv_cara, tolerance_cara, endowment_cara, borrowing_limit=False)

### 2. Tools

Nothing to fill in. Every tool is from an earlier notebook.

| Tool | From | What it does |
| --- | --- | --- |
| `prices` | session 10 | the firm's first-order conditions: $R_t$ and $w_t$ from $K_t$ and $Z_t$ |
| `jacobian`, `linearize` | session 9 | $\mathbf{A}$ and $\mathbf{B}$ of $\mathbf{A} \, \mathbb{E}_t \hat{\mathbf{x}}_{t+1} = \mathbf{B} \hat{\mathbf{x}}_t$ by central differences |
| `solve_klein`, `impulse_response` | sessions 9 and 10 | $\mathbf{G}$, $\mathbf{H}$, the eigenvalues, and the paths after a shock |
| `make_grids`, `interp_extrap` | sessions 5, 6, 9 | Rouwenhorst for $e$, the labor endowments, the asset grid (dense around $a = 0$), interpolation with linear extrapolation |
| `lottery`, `transition_matrix`, `newborns`, `forward`, `stationary_dist` | sessions 6 and 10 | Young's method with newborns |

In [ ]:
def prices(K, Z, par):
    # The firm's first-order conditions: the gross return R = 1 + r and the wage w
    R = par.alpha * Z * K ** (par.alpha - 1) + 1 - par.delta
    w = (1 - par.alpha) * Z * K ** par.alpha
    return R, w


def jacobian(f, x0, h=1e-6):
    # The Jacobian of f at x0 by central differences: column j is df / dx_j (session 9)
    n, m = len(x0), len(f(x0))
    J = np.zeros((m, n))
    for j in range(n):
        e = np.zeros(n)
        e[j] = h
        J[:, j] = (f(x0 + e) - f(x0 - e)) / (2 * h)
    return J


def linearize(F, xss, par):
    # The matrices A and B of the linear system A E x' = B x at the steady state xss (session 9)
    A = jacobian(lambda x_next: F(x_next, xss, par), xss)
    B = -jacobian(lambda x: F(xss, x, par), xss)
    return A, B


def solve_klein(A, B, n_states):
    # Klein (2000) in real arithmetic (session 10): G maps states into controls, H is the law of motion of the states
    S, T, alpha, beta, Q, Z = ordqz(A, B, sort=lambda a, b: np.abs(b) < np.abs(a), output="real")
    n_stable = np.sum(np.abs(beta) < np.abs(alpha))
    if n_stable != n_states:
        print(f"Klein: {n_stable} stable eigenvalues for {n_states} states")
    Z11, Z21 = Z[:n_states, :n_states], Z[n_states:, :n_states]
    S11, T11 = S[:n_states, :n_states], T[:n_states, :n_states]
    G = Z21 @ inv(Z11)                                                             # the controls on the states
    H = Z11 @ solve(S11, T11) @ inv(Z11)                                           # the law of motion of the states
    with np.errstate(divide="ignore", invalid="ignore"):
        eigs = beta / alpha                                                        # our lambda_i = t_ii / s_ii
    return G, H, eigs


def impulse_response(G, H, n_states, T, shock):
    # The paths of states and controls for T periods after the shock vector hits the states in period 0 (session 9)
    s = np.zeros((T, n_states))
    s[0] = shock
    for t in range(T - 1):
        s[t + 1] = H @ s[t]
    return s, s @ G.T

In [ ]:
def make_grids(par, mpar, hh):
    # Rouwenhorst for e (session 5): qe.rouwenhorst wants the standard deviation of the innovation, sig_e sqrt(1 - rho_e^2).
    # The asset grid is quadratically spaced around zero, where the newborns start: dense at a = 0, coarse at amin and amax.
    # With amin = 0 it is the grid of session 6
    mc = qe.rouwenhorst(mpar.ne, par.rho_e, par.sig_e * np.sqrt(1 - par.rho_e ** 2))
    lam_e = mc.stationary_distributions[0]
    s = np.linspace(-np.sqrt(-mpar.amin / mpar.amax), 1.0, mpar.na)
    a = mpar.amax * s * np.abs(s)
    return Grids(a=a, e=mc.state_values, y=hh.endowment(mc.state_values, lam_e), Pi=mc.P, lam_e=lam_e)


def interp_extrap(x, xp, fp):
    # np.interp, but linear extrapolation beyond the ends of the grid instead of a flat continuation (session 9)
    y = np.interp(x, xp, fp)
    lo, hi = x < xp[0], x > xp[-1]
    y[lo] = fp[0] + (x[lo] - xp[0]) * (fp[1] - fp[0]) / (xp[1] - xp[0])
    y[hi] = fp[-1] + (x[hi] - xp[-1]) * (fp[-1] - fp[-2]) / (xp[-1] - xp[-2])
    return y

In [ ]:
def lottery(a_prime, gri):
    # For every policy value a': the index l of the grid point just below it, and the weight w on the point above (session 6)
    na = len(gri.a)
    l = np.clip(np.searchsorted(gri.a, a_prime, side="right") - 1, 0, na - 2)
    w = np.clip((a_prime - gri.a[l]) / (gri.a[l + 1] - gri.a[l]), 0.0, 1.0)
    return l, w


def transition_matrix(a_prime, gri):
    # The policy chain Q of session 6: Q[(i, m), (l, n)] = Pi[m, n] (1 - w_im) and Q[(i, m), (l + 1, n)] = Pi[m, n] w_im
    na, ne = len(gri.a), len(gri.e)
    idx = np.arange(na * ne).reshape(ne, na).T                                     # idx[i, m] = i + m N_a
    l, w = lottery(a_prime, gri)
    rows, cols, vals = [], [], []
    for m in range(ne):
        for n in range(ne):
            rows.append(idx[:, m]); cols.append(idx[l[:, m], n]); vals.append(gri.Pi[m, n] * (1 - w[:, m]))
            rows.append(idx[:, m]); cols.append(idx[l[:, m] + 1, n]); vals.append(gri.Pi[m, n] * w[:, m])
    return sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(na * ne, na * ne))


def newborns(gri):
    # The newborns as a histogram of mass one: a = 0 by the lottery, e from its stationary distribution (session 10)
    l, w = lottery(np.zeros((1, 1)), gri)
    lam_born = np.zeros((len(gri.a), len(gri.e)))
    lam_born[l[0, 0]] = (1 - w[0, 0]) * gri.lam_e
    lam_born[l[0, 0] + 1] = w[0, 0] * gri.lam_e
    return lam_born


def forward(lam, Q, lam_born, par):
    # One period of the histogram: survivors move along the policy chain, the dead are replaced by newborns (session 10)
    na, ne = lam.shape
    survivors = Q.T @ lam.T.ravel()
    lam_new = par.phi * survivors + (1 - par.phi) * lam.sum() * lam_born.T.ravel()
    return lam_new.reshape(ne, na).T


def stationary_dist(Q, lam_born, par, mpar, lam0=None):
    # Iterate forward until the histogram stops moving (session 6), from the newborns or from lam0
    lam = lam_born.copy() if lam0 is None else lam0
    for it in range(1, mpar.maxiter + 1):
        lam_new = forward(lam, Q, lam_born, par)
        dist = np.max(np.abs(lam_new - lam))
        lam = lam_new
        if dist < mpar.crit_dist:
            break
    return lam, it

### 3. The aggregate economy: a PRANC with five equations

This is the system of Part 1, in $\mathbf{x}_t = (K_t, Z_t, C_t, \mu_t, \nu_t)$, states first:

$$
\begin{aligned}
K_{t+1} &= Z_t K_t^\alpha + (1 - \delta) K_t - C_t && \text{resource constraint} \\
\ln Z_{t+1} - \ln \bar Z &= \rho_Z (\ln Z_t - \ln \bar Z) && \text{TFP} \\
C_{t+1} &= C_t + \tfrac{1}{\gamma} \ln (\beta R_{t+1}) - \tfrac{1 - \varphi}{\varphi} \mu_{t+1} R_{t+1} K_{t+1}
          + \tfrac{\gamma}{2} (\mu_{t+1} + \nu_{t+1})^2 w_{t+1}^2 \sigma_\varepsilon^2 && \text{aggregate Euler equation} \\
\mu_t &= \frac{\mu_{t+1} R_{t+1}}{\varphi + \mu_{t+1} R_{t+1}} && \text{the MPC} \\
\nu_t &= \rho_e (1 - \mu_t) (\mu_{t+1} + \nu_{t+1}) \frac{w_{t+1}}{w_t} && \text{the loading on } e
\end{aligned}
$$

with $\sigma_\varepsilon^2 = \sigma_e^2 (1 - \rho_e^2)$. The last equation is the one of the problem set. The
session 10 slide has $\varphi / R_{t+1}$ in the place of $1 - \mu_t$, which holds in the steady state only.

`solve_pranc(par)` does everything: it **calibrates** $\beta$ and $\bar Z$ so that the steady state has $r = 4$
percent and $Y = 1$, whatever the other parameters in `par` are, linearizes, solves with Klein, and returns the
impulse responses to a one percent TFP shock. Its result `res` is a dictionary: `res["par"]` is the calibrated
`par`, `res["ss"]` the steady state, `res["K"]`, `res["C"]`, `res["Y"]`, `res["Z"]`, `res["mu"]`, `res["nu"]`
the paths in log deviations, `res["eigs"]` the eigenvalues.

In [ ]:
def pranc_steady_state(par, r=0.04, Y=1.0):
    # The steady state of the aggregate economy with the rate r and output Y: the firm gives K, Zbar, w,
    # the household's coefficients mu and nu, and the aggregate Euler equation with C' = C gives beta
    R = 1 + r
    K = par.alpha * Y / (r + par.delta)                                            # the firm: r = alpha Y / K - delta
    Zbar = Y / K ** par.alpha
    w = (1 - par.alpha) * Y
    C = Y - par.delta * K                                                          # the resource constraint, K' = K
    mu = 1 - par.phi / R                                                           # the MPC out of cash on hand
    nu = par.rho_e * par.phi * mu / (R - par.rho_e * par.phi)                      # the extra response to a persistent e
    sig_eps2 = par.sig_e ** 2 * (1 - par.rho_e ** 2)                               # variance of the innovation to e
    newborn = (1 - par.phi) / par.phi * mu * R * K                                 # the term of the newborns
    precaution = par.gamma / 2 * (mu + nu) ** 2 * w ** 2 * sig_eps2                # the precautionary term
    beta = float(np.exp(par.gamma * (newborn - precaution)) / R)
    return dict(K=K, C=C, Z=Zbar, R=R, w=w, mu=mu, nu=nu, beta=beta, Y=Y, newborn=newborn, precaution=precaution)


def F_pranc(x_next, x, par):
    # The five equations as residuals. x = (ln K, ln Z, ln C, ln mu, ln nu) today, x_next tomorrow
    K, Z, C, mu, nu = np.exp(x)
    K1, Z1, C1, mu1, nu1 = np.exp(x_next)
    R, w = prices(K, Z, par)
    R1, w1 = prices(K1, Z1, par)
    sig_eps2 = par.sig_e ** 2 * (1 - par.rho_e ** 2)
    return np.array([
        np.log(K1) - np.log(Z * K ** par.alpha + (1 - par.delta) * K - C),        # resource constraint
        np.log(Z1 / par.Zbar) - par.rho_Z * np.log(Z / par.Zbar),                  # TFP, without the shock
        C1 - C - np.log(par.beta * R1) / par.gamma                                 # aggregate Euler equation:
        + (1 - par.phi) / par.phi * mu1 * R1 * K1                                  #   the newborns own none of K'
        - par.gamma / 2 * (mu1 + nu1) ** 2 * w1 ** 2 * sig_eps2,                   #   precaution
        np.log(mu) - np.log(mu1 * R1 / (par.phi + mu1 * R1)),                      # the MPC
        np.log(nu) - np.log(par.rho_e * (1 - mu) * (mu1 + nu1) * w1 / w),          # the loading on e
    ])

In [ ]:
def solve_pranc(par, T=60, shock=0.01, verbose=True):
    # The aggregate economy from start to end: calibration, linearization, Klein, impulse responses to a TFP shock
    ss = pranc_steady_state(par)
    par = replace(par, beta=ss["beta"], Zbar=ss["Z"])                              # the calibrated economy
    xss = np.log([ss["K"], ss["Z"], ss["C"], ss["mu"], ss["nu"]])
    A, B = linearize(F_pranc, xss, par)
    G, H, eigs = solve_klein(A, B, 2)
    s, c = impulse_response(G, H, 2, T, np.array([0.0, shock]))
    res = dict(name="PRANC", par=par, ss=ss, G=G, H=H, eigs=eigs, K=s[:, 0], Z=s[:, 1], Y=s[:, 1] + par.alpha * s[:, 0],
               C=c[:, 0], mu=c[:, 1], nu=c[:, 2])
    if verbose:
        print(f"PRANC with {par}")
        print(f"  steady state: K = {ss['K']:.4f}, C = {ss['C']:.4f}, mu = {ss['mu']:.5f}, nu = {ss['nu']:.5f}, "
              f"beta = {ss['beta']:.5f}, beta R = {ss['beta'] * ss['R']:.5f}")
        print(f"  largest residual at the steady state: {np.max(np.abs(F_pranc(xss, xss, par))):.1e}, "
              f"eigenvalues: {np.sort(np.abs(eigs))}")
        print(f"  C on impact {100 * res['C'][0]:.3f} percent, K peaks at {100 * res['K'].max():.3f} percent in period {res['K'].argmax()}")
    return res

### 4. The heterogeneous agent economy: Reiter's method

The functions of session 10, with one change: the household is the argument `hh`. Read `egm_step` with the two
models in mind: the Euler equation is inverted with `hh.uprime_inv`, the envelope condition uses `hh.uprime`,
income is the wage times the labor endowment `gri.y`, and one line imposes the borrowing limit if the household
has one.

`solve_reiter(par, mpar, hh)` does everything for a calibrated `par`: the grids, the steady state by bisection
on $r$, the system $\mathbf{F}$ with $2 N_a N_e + 1$ variables, its linearization, Klein, and the impulse
responses to a one percent TFP shock. Its result `res` is a dictionary: `res["K"]`, `res["C"]`, `res["Y"]`,
`res["Z"]` are the paths in log deviations, `res["agg"]` the steady state on the grid (`agg.c`, `agg.a_prime`,
`agg.lam`, `agg.K`, ...), `res["gri"]` the grids, `res["r_star"]`, `res["eigs"]`, `res["resid"]`, and the run
times `res["t_ss"]`, `res["t_jac"]`, `res["t_qz"]`.

In [ ]:
def egm_step(lnv_next, R, w, par, gri, hh):
    # One step of the EGM on marginal values (session 10): from ln v' on the grid (a'_l, e_n) to today's savings policy
    # a_prime, consumption c, and ln v, all N_a x N_e, at today's prices R and w
    Ev = par.phi * np.exp(lnv_next) @ gri.Pi.T                                    # E[v' | e_m] at every a'_l, the dead need no assets
    ctilde = hh.uprime_inv(par.beta * Ev, par)                                     # (a) the Euler equation u'(c) = beta phi E[v'], solved for c
    xtilde = ctilde + gri.a[:, None]                                               # (b) the cash on hand at which a'_l is chosen
    x = R / par.phi * gri.a[:, None] + w * gri.y[None, :]                    # cash on hand on the grid
    a_prime = np.empty_like(x)
    for m in range(len(gri.e)):                                                    # (c) back to the grid, one income state at a time
        a_prime[:, m] = interp_extrap(x[:, m], xtilde[:, m], gri.a)
    if hh.borrowing_limit:
        a_prime = np.maximum(a_prime, gri.a[0])                                    # the borrowing limit a' >= a_1
    c = x - a_prime
    lnv = np.log(R / par.phi) + np.log(hh.uprime(c, par))                          # (d) the envelope condition v = (R / phi) u'(c), in logs
    return a_prime, c, lnv


def solve_household(R, w, par, mpar, gri, hh, lnv0=None):
    # Iterate egm_step at constant prices until ln v stops moving (session 5). The guess: keep assets constant, a' = a
    if lnv0 is None:
        x = R / par.phi * gri.a[:, None] + w * gri.y[None, :]
        lnv0 = np.log(R / par.phi) + np.log(hh.uprime(x - gri.a[:, None], par))
    lnv = lnv0
    for it in range(1, mpar.maxiter + 1):
        a_prime, c, lnv_new = egm_step(lnv, R, w, par, gri, hh)
        dist = np.max(np.abs(lnv_new - lnv))
        lnv = lnv_new
        if dist < mpar.crit_pol:
            break
    return a_prime, c, lnv, it

In [ ]:
@dataclass
class Aggregates:
    S: float                  # asset supply: mean savings of the households
    K: float                  # capital demand of the firm
    R: float                  # gross return 1 + r
    w: float                  # wage
    a_prime: np.ndarray       # savings policy a'(a, e), N_a x N_e
    c: np.ndarray             # consumption policy
    lnv: np.ndarray           # log marginal values
    lam: np.ndarray           # stationary histogram


def aggregate(r, par, mpar, gri, hh, lam_born, warm=None):
    # The household block at the rate r (sessions 7 and 8): prices, policy, histogram, supply S(r) and demand K(r)
    K = (par.alpha * par.Zbar / (r + par.delta)) ** (1 / (1 - par.alpha))          # the firm's capital demand at r
    R, w = prices(K, par.Zbar, par)
    a_prime, c, lnv, _ = solve_household(R, w, par, mpar, gri, hh, None if warm is None else warm.lnv)
    lam, _ = stationary_dist(transition_matrix(a_prime, gri), lam_born, par, mpar, None if warm is None else warm.lam)
    return Aggregates(S=(lam * a_prime).sum(), K=K, R=R, w=w, a_prime=a_prime, c=c, lnv=lnv, lam=lam)


def equilibrium(par, mpar, gri, hh, lam_born, lo, hi, tol=1e-12):
    # Bisection on the excess supply S(r) - K(r), which increases in r, with warm starts (session 8)
    warm, k = None, 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        warm = aggregate(mid, par, mpar, gri, hh, lam_born, warm)
        lo, hi = (mid, hi) if warm.S < warm.K else (lo, mid)
        k += 1
    r = (lo + hi) / 2
    return r, aggregate(r, par, mpar, gri, hh, lam_born, warm), k

In [ ]:
def unpack(x, gri):
    # The vector x as its four blocks: the histogram (with its last bin restored), K, Z, and ln v (session 10)
    na, ne = len(gri.a), len(gri.e)
    N = na * ne
    lam_vec = np.append(x[:N - 1], 1 - x[:N - 1].sum())                            # the last bin: the histogram sums to one
    lam = lam_vec.reshape(ne, na).T
    lnv = x[N + 1:].reshape(ne, na).T
    return lam, np.exp(x[N - 1]), np.exp(x[N]), lnv


def pack(lam, K, Z, lnv):
    # The inverse of unpack: the four blocks as one vector, the histogram without its last bin
    N = lam.size
    return np.concatenate([lam.T.ravel()[:N - 1], [np.log(K), np.log(Z)], lnv.T.ravel()])


def F_reiter(x_next, x, par, gri, hh, lam_born):
    # Reiter's system (session 10): zero along an equilibrium path without shocks
    lam, K, Z, lnv = unpack(x, gri)                                                # today
    lam1, K1, Z1, lnv1 = unpack(x_next, gri)                                       # tomorrow
    R, w = prices(K, Z, par)                                                       # today's prices
    a_prime, c, lnv_egm = egm_step(lnv1, R, w, par, gri, hh)                       # one EGM step from tomorrow's v
    lam_fwd = forward(lam, transition_matrix(a_prime, gri), lam_born, par)         # one step of Young's method
    N = lam.size
    return np.concatenate([
        (lam1 - lam_fwd).T.ravel()[:N - 1],                                        # Young: all bins but the last
        [np.log(K1) - np.log((lam * a_prime).sum())],                              # asset market: K' is the mean of today's savings
        [np.log(Z1 / par.Zbar) - par.rho_Z * np.log(Z / par.Zbar)],                # TFP, without the shock
        (lnv - lnv_egm).T.ravel(),                                                 # EGM: today's marginal values
    ])


def consumption_from_households(s_h, c_h, agg, par, gri, hh):
    # C_t = sum lam_t c_t, linearized: d lam times c plus lam times dc. The change of the policy is read off the
    # marginal values: ln v = ln(R / phi) + ln u'(c), so dc = tolerance(c) (d ln R - d ln v)
    na, ne = len(gri.a), len(gri.e)
    N = na * ne
    dlnR = (agg.R - 1 + par.delta) / agg.R * (s_h[:, N] + (par.alpha - 1) * s_h[:, N - 1])   # d ln R from d ln Z and d ln K
    dC = np.zeros(len(s_h))
    for t in range(len(s_h)):
        dlam = np.append(s_h[t, :N - 1], -s_h[t, :N - 1].sum()).reshape(ne, na).T  # the change of the histogram, sums to zero
        dlnv = c_h[t].reshape(ne, na).T                                            # the change of ln v on the grid
        dc = hh.tolerance(agg.c, par) * (dlnR[t] - dlnv)
        dC[t] = (dlam * agg.c).sum() + (agg.lam * dc).sum()
    return dC / (agg.lam * agg.c).sum()

In [ ]:
def solve_reiter(par, mpar, hh, T=60, shock=0.01, r_lo=0.03, r_hi=0.05, verbose=True):
    # The heterogeneous agent economy from start to end: grids, steady state, Reiter's system, Klein, impulse responses
    gri = make_grids(par, mpar, hh)
    lam_born = newborns(gri)
    t0 = time.perf_counter()
    r_star, agg, _ = equilibrium(par, mpar, gri, hh, lam_born, r_lo, r_hi)        # the steady state on the grid
    t_ss = time.perf_counter() - t0
    N = mpar.na * mpar.ne
    n_states = N + 1                                                               # the histogram without its last bin, K, and Z
    xss = pack(agg.lam, agg.K, par.Zbar, agg.lnv)
    F = lambda x_next, x, par: F_reiter(x_next, x, par, gri, hh, lam_born)         # F with the grids and the household filled in
    resid = np.max(np.abs(F(xss, xss, par)))
    t0 = time.perf_counter()
    A, B = linearize(F, xss, par)
    t_jac = time.perf_counter() - t0
    t0 = time.perf_counter()
    G, H, eigs = solve_klein(A, B, n_states)
    t_qz = time.perf_counter() - t0
    s0 = np.zeros(n_states)
    s0[N] = shock                                                                  # log TFP is the last state
    s_h, c_h = impulse_response(G, H, n_states, T, s0)
    K_h, Z_h = s_h[:, N - 1], s_h[:, N]                                            # log deviations of K and Z
    Y_h = Z_h + par.alpha * K_h
    C_h = consumption_from_households(s_h, c_h, agg, par, gri, hh)                 # consumption summed over the households
    C, Y = (agg.lam * agg.c).sum(), par.Zbar * agg.K ** par.alpha
    C_rc = (Y * Y_h[:-1] + (1 - par.delta) * agg.K * K_h[:-1] - agg.K * K_h[1:]) / C   # ... and from the resource constraint
    at_limit = (agg.lam * (agg.a_prime <= gri.a[0] + 1e-10)).sum()                 # share of households with a' at the lowest grid point
    res = dict(name=f"Reiter ({hh.name})", par=par, mpar=mpar, hh=hh, gri=gri, lam_born=lam_born, r_star=r_star, agg=agg,
               G=G, H=H, eigs=eigs, K=K_h, Z=Z_h, Y=Y_h, C=C_h, resid=resid, walras=np.max(np.abs(C_h[:-1] - C_rc)),
               at_limit=at_limit, top=agg.lam[-1].sum(), bottom=agg.lam[0].sum(), t_ss=t_ss, t_jac=t_jac, t_qz=t_qz)
    if verbose:
        mods = np.abs(eigs)
        print(f"Reiter ({hh.name}) with {par}, N_a = {mpar.na}, N_e = {mpar.ne}, grid = [{mpar.amin}, {mpar.amax}]")
        print(f"  steady state: r* = {100 * r_star:.5f} percent, K = {agg.K:.5f}, C = {C:.5f}, mass at a_1: {res['bottom']:.1e}, "
              f"at a_N: {res['top']:.1e}, households with a' = a_1: {100 * at_limit:.2f} percent ({t_ss:.1f} s)")
        print(f"  Reiter: {len(xss)} variables, largest residual at the steady state {resid:.1e}, {np.sum(mods < 1)} stable eigenvalues "
              f"for {n_states} states, Jacobians {t_jac:.1f} s, Klein {t_qz:.1f} s")
        print(f"  C on impact {100 * C_h[0]:.3f} percent, K peaks at {100 * K_h.max():.3f} percent in period {K_h.argmax()}, "
              f"goods market gap {res['walras']:.1e}")
    return res

### 5. Reporting

| Tool | What it does |
| --- | --- |
| `compare(results)` | one line per result: consumption on impact, the peak of capital and its period |
| `plot_irfs(results)` | capital, consumption, and output after the shock, one line per result |
| `plot_cdf(results, ss)` | CARA only: the distribution of cash on hand on the grid, one line per result, against the analytical CDF of session 10 |

`results` is a dictionary `{label: res}` of results of `solve_pranc`, `solve_reiter`, or `solve_rbc`.

In [ ]:
def compare(results):
    # One line per result: the two numbers we compare across economies
    print(f"{'':36s}{'C on impact':>12s}{'K at peak':>12s}{'period':>8s}")
    for label, res in results.items():
        print(f"{label:36s}{100 * res['C'][0]:12.3f}{100 * res['K'].max():12.3f}{res['K'].argmax():8d}")


def plot_irfs(results, title=None):
    # Capital, consumption, and output in percent deviation from the steady state, one line per result
    styles = ["-", "--", ":", "-."]
    fig, axs = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
    for ax, (key, name) in zip(axs, (("K", "capital"), ("C", "consumption"), ("Y", "output"))):
        for k, (label, res) in enumerate(results.items()):
            ax.plot(100 * res[key], styles[k % 4], lw=2, label=label)
        ax.set_title(name, fontsize=10); ax.set_xlabel("periods after the shock"); ax.axhline(0, color="k", lw=0.5)
    axs[0].set_ylabel("percent deviation"); axs[0].legend(fontsize=8)
    if title is not None:
        fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

In [ ]:
def analytical_cdf(x, par, ss, tol=1e-13):
    # The stationary CDF of cash on hand in the CARA economy (session 10): a mixture over ages k of normals N(m_k, V_k)
    Rphi, w, mu, nu = ss["R"] / par.phi, ss["w"], ss["mu"], ss["nu"]
    f = ss["C"] - mu * (ss["R"] * ss["K"] + w)                                     # the intercept of the policy
    kappa = par.rho_e - Rphi * nu                                                  # how much of e stays in x
    sig_eps2 = par.sig_e ** 2 * (1 - par.rho_e ** 2)
    m, V, Cov = w, w ** 2 * par.sig_e ** 2, w * par.sig_e ** 2                     # age 0: x = w (1 + e)
    d = w - Rphi * f                                                               # the mean grows by d per year of age
    F, weight = np.zeros_like(x), 1 - par.phi
    while weight > tol:
        F += weight * norm.cdf((x - m) / np.sqrt(V))
        V, Cov = (V + w ** 2 * kappa ** 2 * par.sig_e ** 2 + w ** 2 * sig_eps2 + 2 * w * kappa * Cov,
                  par.rho_e * Cov + w * (par.rho_e * kappa * par.sig_e ** 2 + sig_eps2))
        m += d
        weight *= par.phi
    return F


def plot_cdf(results, ss):
    # CARA only: the CDF of cash on hand on the grid, one line per result, against the analytical one
    plt.figure(figsize=(6, 3))
    for label, res in results.items():
        agg, gri, par = res["agg"], res["gri"], res["par"]
        x = agg.R / par.phi * gri.a[:, None] + agg.w * gri.y[None, :]
        order = np.argsort(x.ravel())
        plt.step(x.ravel()[order], np.cumsum(agg.lam.ravel()[order]), where="post", lw=1.5, label=label)
    xx = np.linspace(-2.0, 15.0, 400)
    plt.plot(xx, analytical_cdf(xx, par, ss), "k--", lw=1, label="analytical")
    plt.xlim(-2, 15); plt.xlabel("cash on hand $x$"); plt.ylabel("CDF"); plt.legend(fontsize=8)
    plt.tight_layout(); plt.show()

### 6. Part 2: the CARA economy

**Question 2.1: five equations against 801 variables.** Two calls. `solve_pranc` calibrates the economy and
returns the calibrated parameters, `solve_reiter` solves the same economy on the grid.

In [ ]:
pranc = solve_pranc(EconomicParameters())
par = pranc["par"]                                                                 # the calibrated economy
ss = pranc["ss"]

In [ ]:
mpar = NumericalParameters()
reiter = solve_reiter(par, mpar, CARA)

In [ ]:
compare({"PRANC, 5 equations": pranc, "Reiter, 801 variables": reiter})
plot_irfs({"Reiter, 801 variables": reiter, "PRANC, 5 equations": pranc})

In [ ]:
plot_cdf({"Reiter, on the grid": reiter}, ss)

**Check:** $\beta = 0.97471$, $\mu = 0.05983$, $\nu = 0.32906$. The PRANC has two stable eigenvalues, $0.900$
and $0.927$, and three explosive ones, $1.051$, $1.132$, $1.184$. On the grid $r^* = 4.00012$ percent, the
residual at the steady state is below $10^{-9}$, and there are $401$ stable eigenvalues for $401$ states. In
both, consumption rises by $0.552$ percent on impact and capital peaks at $1.026$ percent in period $11$: the
lines of the two solvers lie on top of each other.

**Question 2.2: varying the grids.** Solve the heterogeneous agent economy with other numbers of grid points,
`solve_reiter(par, NumericalParameters(na=40), CARA)` for the asset grid and `NumericalParameters(ne=3)` for the
income grid, and so on. The run times are `res["t_ss"]`, `res["t_jac"]`, and `res["t_qz"]`. `plot_cdf` and
`plot_irfs` take several results at once.

In [ ]:
# Your code for Question 2.2

**Question 2.3: income risk.** Change a parameter and solve again: `solve_pranc(EconomicParameters(sig_e=0.3))`.
Every call of `solve_pranc` calibrates $\beta$ and $\bar Z$ anew, so that all economies have $r = 4$ percent and
$Y = 1$ in the steady state. Put the results side by side with `compare` and `plot_irfs`.

In [ ]:
# Your code for Question 2.3

### 7. Part 3: the CRRA economy with a borrowing limit

Three changes to the economy of Part 2. Utility is $u(c) = c^{1 - \gamma} / (1 - \gamma)$ with $\gamma = 3$.
Income follows an AR(1) in **logs**: $e$ is the same process as before, but the labor endowment is $y(e) =
\exp(e) / \mathbb{E}[\exp(e)]$ in the place of $1 + e$, so that income $w_t \, y(e_t)$ is positive and has mean
$w_t$. And households cannot borrow, $a' \geq 0$. Everything else stays: the survival probability, the
annuities, the newborns with $a = 0$, the firm, and $\bar Z$. The solvers of section 4 stay as well. What
changes is the household `hh` that you pass to them, the parameters, how the steady state is calibrated, and the
benchmark. Those are the gaps.

**Gaps 1 to 3: the household.** Marginal utility and its inverse for CRRA utility, and the labor endowment.
$-u'(c) / u''(c)$ and the borrowing limit are given.

In [ ]:
def uprime_crra(c, par):
    return ...  # GAP 1: marginal utility u'(c) of CRRA utility


def uprime_inv_crra(m, par):
    return ...  # GAP 2: the inverse of marginal utility, the c with u'(c) = m


def tolerance_crra(c, par):
    # -u'(c) / u''(c) for CRRA utility (CARA: 1 / gamma)
    return c / par.gamma


def endowment_crra(e, lam_e):
    # e is the vector of income states, lam_e its stationary distribution
    return ...  # GAP 3: the labor endowment exp(e), divided by its mean over lam_e (CARA: 1 + e)


CRRA = Household("CRRA", uprime_crra, uprime_inv_crra, tolerance_crra, endowment_crra,
                 borrowing_limit=True)                                             # the lowest grid point is a constraint of the household

**Gaps 4 and 5: parameters and grid.** Relative risk aversion $\gamma = 3$: at the mean consumption of the CARA
economy, $C = 0.743$, the CARA household with $\gamma = 4$ has a relative risk aversion of $\gamma C = 2.97$.
The asset grid starts at the borrowing limit and reaches further up than in Part 2.

In [ ]:
par3 = EconomicParameters(gamma=...,  # GAP 4: relative risk aversion
                          Zbar=ss["Z"])                                            # the same technology as in Part 2, beta is calibrated below
mpar3 = NumericalParameters(amin=...,  # GAP 5: the lowest asset grid point is the borrowing limit
                            amax=300.0)

**The calibration.** There is no closed form for the steady state, so $\beta$ cannot be computed from a formula
as in `pranc_steady_state`. `calibrate_beta` finds, by bisection, the $\beta$ at which the households supply
exactly the capital that the firm demands at $r = 4$ percent. The steady state then has the same $r$, $K$, $Y$,
and $C$ as the CARA economy.

In [ ]:
def calibrate_beta(par, mpar, hh, r=0.04, lo=0.90, hi=0.99, tol=1e-13):
    # The beta at which the asset market clears at the rate r: bisection on beta, with warm starts
    gri = make_grids(par, mpar, hh)
    lam_born = newborns(gri)
    warm = None
    while hi - lo > tol:
        mid = (lo + hi) / 2
        warm = aggregate(r, replace(par, beta=mid), mpar, gri, hh, lam_born, warm)
        lo, hi = (mid, hi) if warm.S < warm.K else (lo, mid)                      # too little saving: households have to be more patient
    return replace(par, beta=(lo + hi) / 2)

**Gaps 6 and 7: the benchmark.** Without a PRANC, the benchmark is the representative agent of session 9: the
RBC model with CRRA utility, here with the technology, the TFP process, and the steady state of this problem
set. Its three equations are the resource constraint, TFP, and its Euler equation. `solve_rbc` returns the same
kind of result as `solve_pranc`.

In [ ]:
def F_rbc(x_next, x, par):
    # The representative agent (session 9). x = (ln K, ln Z, ln C) today, x_next tomorrow
    K, Z, C = np.exp(x)
    K1, Z1, C1 = np.exp(x_next)
    R1, _ = prices(K1, Z1, par)
    return np.array([
        np.log(K1) - np.log(Z * K ** par.alpha + (1 - par.delta) * K - C),        # resource constraint
        np.log(Z1 / par.Zbar) - par.rho_Z * np.log(Z / par.Zbar),                  # TFP, without the shock
        # GAP 6: the Euler equation of the representative agent with CRRA utility, as a residual in logs
        ...,
    ])


def solve_rbc(par, ss, T=60, shock=0.01, verbose=True):
    # The representative agent economy with the steady state ss of the PRANC: linearization, Klein, impulse responses
    par = replace(par, beta=...)  # GAP 7: the beta of the representative agent in a steady state with the gross rate ss["R"]
    xss = np.log([ss["K"], ss["Z"], ss["C"]])
    A, B = linearize(F_rbc, xss, par)
    G, H, eigs = solve_klein(A, B, 2)
    s, c = impulse_response(G, H, 2, T, np.array([0.0, shock]))
    res = dict(name="RBC", par=par, ss=ss, G=G, H=H, eigs=eigs, K=s[:, 0], Z=s[:, 1], Y=s[:, 1] + par.alpha * s[:, 0], C=c[:, 0])
    if verbose:
        print(f"RBC with {par}")
        print(f"  largest residual at the steady state: {np.max(np.abs(F_rbc(xss, xss, par))):.1e}, eigenvalues: {np.sort(np.abs(eigs))}")
        print(f"  C on impact {100 * res['C'][0]:.3f} percent, K peaks at {100 * res['K'].max():.3f} percent in period {res['K'].argmax()}")
    return res

**Question 3.1: run it.** Three calls: the calibration, Reiter's method, and the benchmark.

In [ ]:
par3 = calibrate_beta(par3, mpar3, CRRA)
print(f"beta = {par3.beta:.5f}, beta R = {1.04 * par3.beta:.5f}")
reiter3 = solve_reiter(par3, mpar3, CRRA)

In [ ]:
rbc3 = solve_rbc(par3, ss)

**Check:** $\beta = 0.97352$, $r^* = 4.00000$ percent, $K = 2.57143$ and $C = 0.74286$ as in Part 2. $1.55$
percent of the households choose $a' = 0$. The residual at the steady state is below $10^{-8}$, and there are
$401$ stable eigenvalues for $401$ states. Consumption rises by $0.544$ percent on impact and capital peaks at
$1.044$ percent in period $11$. In the representative agent economy the eigenvalues are $0.900$, $0.930$, and
$1.118$, consumption rises by $0.545$ percent and capital peaks at $1.051$ percent. If the residual at the
steady state is large, check gaps 1 to 3.

**Question 3.2: the steady state.** The figure: the consumption policy and the distribution of assets.

In [ ]:
agg3, gri3 = reiter3["agg"], reiter3["gri"]
fig, axs = plt.subplots(1, 2, figsize=(8, 3.2))
for m, label in ((0, "lowest $e$"), (2, "$e = 0$"), (4, "highest $e$")):
    axs[0].plot(gri3.a, agg3.c[:, m], lw=2, label=label)
axs[0].set_xlim(0, 10); axs[0].set_ylim(0.4, 1.4); axs[0].set_xlabel("assets $a$"); axs[0].set_ylabel("consumption"); axs[0].legend(fontsize=8)
axs[1].plot(gri3.a, np.cumsum(agg3.lam.sum(axis=1)), lw=2)
axs[1].set_xlim(0, 25); axs[1].set_xlabel("assets $a$"); axs[1].set_ylabel("share of households with less")
plt.tight_layout(); plt.show()

**Question 3.3: the dynamics.** The CRRA economy and its representative agent side by side.

In [ ]:
compare({"CRRA, Reiter": reiter3, "CRRA, representative agent": rbc3})
plot_irfs({"CRRA, Reiter": reiter3, "CRRA, representative agent": rbc3})

**Question 3.4: income risk.** Repeat the calls of Question 3.1 for the economies that the question asks for.
Each needs its own calibration: start from `replace(par3, sig_e=...)`, and call `calibrate_beta` again.

In [ ]:
# Your code for Question 3.4